In [0]:
import re
from pyspark.sql import functions as F

base = "/Volumes/workspace/default/raw"

raw = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("header", True).option("multiLine", True).option("escape", '"')
       .option("cloudFiles.schemaLocation", f"{base}/_schema")
       .load(f"{base}/incoming/"))

# same snake_case rename as bronze, plus lineage columns
raw = raw.toDF(*[re.sub(r'[^a-z0-9]+', '_', c.strip().lower()).strip('_') for c in raw.columns])
raw = (raw.withColumn("source_file", F.col("_metadata.file_path"))
          .withColumn("ingest_ts", F.current_timestamp()))

(raw.writeStream
    .option("checkpointLocation", f"{base}/_checkpoint")
    .option("mergeSchema", "true")          # allows the new lineage columns
    .trigger(availableNow=True)             # load all new files, then stop
    .toTable("bronze_complaints")
    .awaitTermination())

display(spark.sql("SELECT source_file, COUNT(*) FROM bronze_complaints GROUP BY source_file"))

from pyspark.sql import Window

new = spark.table("bronze_complaints").filter(F.col("source_file").isNotNull())

text_cols = ["product","sub_product","issue","sub_issue","company","state",
             "submitted_via","company_response_to_consumer","company_public_response","tags"]
for c in text_cols:
    new = new.withColumn(c, F.when(F.trim(F.col(c)).isin("", "None", "N/A"), None)
                             .otherwise(F.trim(F.col(c))))

new = (new
    .withColumn("complaint_id", F.col("complaint_id").cast("bigint"))
    .withColumn("date_received", F.to_date("date_received"))
    .withColumn("date_sent_to_company", F.to_date("date_sent_to_company"))
    .withColumn("timely_response_flag", F.when(F.col("timely_response")=="Yes", True)
                                         .when(F.col("timely_response")=="No", False))
    .withColumn("company", F.upper("company"))
    .withColumn("state", F.upper("state"))
    .withColumn("days_to_company", F.datediff("date_sent_to_company", "date_received"))
    .withColumn("load_ts", F.current_timestamp())
    .drop("source_file", "ingest_ts", "_rescued_data"))

w = Window.partitionBy("complaint_id").orderBy(F.col("date_received").desc())
new = new.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

new.createOrReplaceTempView("daily_batch")
print("Rows in batch:", new.count())




In [0]:
%sql 
MERGE INTO silver_complaints AS t
USING daily_batch AS s
  ON t.complaint_id = s.complaint_id
WHEN MATCHED AND (t.company_response_to_consumer <=> s.company_response_to_consumer) = false
             OR (t.timely_response <=> s.timely_response) = false
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *